In [ ]:
!pip install -q sentence-transformers transformers datasets

from google.colab import drive
drive.mount("/content/drive")

import os
SAVE_DIR = "/content/drive/MyDrive/hallucination_week3"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving to:", SAVE_DIR)

Mounted at /content/drive
Saving to: /content/drive/MyDrive/hallucination_week3


In [ ]:
from datasets import load_dataset
import random, json
from collections import Counter

ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")["evaluation"]

ids = list(range(len(ds)))
random.Random(42).shuffle(ids)          # fixed seed, so the split never changes
tuning_ids = sorted(ids[:80])
test_ids = sorted(ids[80:])

json.dump({"tuning": tuning_ids, "test": test_ids}, open(f"{SAVE_DIR}/split.json", "w"))

def describe(name, idx_list):
    labels = [l for i in idx_list for l in ds[i]["annotation"]]
    n = len(labels)
    hall = sum(l != "accurate" for l in labels)
    p = hall / n
    f1 = 2 * p / (p + 1)   # baseline that says "hallucinated" for everything
    print(f"{name}: {len(idx_list)} passages, {n} sentences, "
          f"{hall} hallucinated ({p:.0%}) | all-hallucinated baseline: precision {p:.2f}, recall 1.00, F1 {f1:.2f}")

describe("TUNING", tuning_ids)
describe("TEST  ", test_ids)

README.md:   0%|          | 0.00/2.45k [00:00<?, ?B/s]

data/evaluation-00000-of-00001-e91191b8f(…): reconstructing file:   0%|          |  0.00B / 2.56MB            

data/evaluation-00000-of-00001-e91191b8f(…): downloading bytes:           |  0.00B            

Generating evaluation split:   0%|          | 0/238 [00:00<?, ? examples/s]

TUNING: 80 passages, 655 sentences, 477 hallucinated (73%) | all-hallucinated baseline: precision 0.73, recall 1.00, F1 0.84
TEST  : 158 passages, 1253 sentences, 915 hallucinated (73%) | all-hallucinated baseline: precision 0.73, recall 1.00, F1 0.84


In [ ]:
import json, re, time, torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline

ds = load_dataset("potsawee/wiki_bio_gpt3_hallucination")["evaluation"]
split = json.load(open(f"{SAVE_DIR}/split.json"))
tuning_ids = split["tuning"]

embedder = SentenceTransformer("all-MiniLM-L6-v2")
nli = pipeline("text-classification",
               model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
               top_k=None,
               device=0 if torch.cuda.is_available() else -1)

print("GPU available:", torch.cuda.is_available())
print("Tuning passages:", len(tuning_ids))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (15 outputs left out)

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

GPU available: True
Tuning passages: 80


In [ ]:
def score_passages(id_list, out_path):
    rows = []
    start = time.time()
    for n, pid in enumerate(id_list):
        ex = ds[pid]
        ev = [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", ex["wiki_bio_text"])
              if len(s.strip()) > 20]
        claims = ex["gpt3_sentences"]
        labels = ex["annotation"]
        if not ev or not claims:
            continue

        ev_emb = embedder.encode(ev, convert_to_tensor=True)
        c_emb = embedder.encode(claims, convert_to_tensor=True)
        sims = util.cos_sim(c_emb, ev_emb)

        pairs, meta = [], []
        for ci, claim in enumerate(claims):
            top = sims[ci].topk(min(5, len(ev)))
            for rank, (s, j) in enumerate(zip(top.values.tolist(), top.indices.tolist())):
                pairs.append({"text": ev[j], "text_pair": claim})
                meta.append((ci, rank, s, ev[j]))

        outs = nli(pairs, batch_size=32, truncation=True)

        per = {}
        for (ci, rank, s, evtxt), o in zip(meta, outs):
            sc = {d["label"].lower(): d["score"] for d in o}
            per.setdefault(ci, []).append({
                "rank": rank, "sim": round(s, 3),
                "ent": round(sc.get("entailment", 0), 3),
                "neu": round(sc.get("neutral", 0), 3),
                "con": round(sc.get("contradiction", 0), 3),
                "evidence": evtxt})

        for ci in range(min(len(claims), len(labels))):
            rows.append({"passage": pid, "sent_idx": ci, "claim": claims[ci],
                         "label": labels[ci], "evidence": per.get(ci, [])})

        if (n + 1) % 10 == 0:
            print(f"{n + 1}/{len(id_list)} passages | {len(rows)} sentences | {round(time.time() - start)} s")

    json.dump(rows, open(out_path, "w"))
    print("Saved", len(rows), "sentences to", out_path, "| total time:", round(time.time() - start), "s")
    return rows

tuning_rows = score_passages(tuning_ids, f"{SAVE_DIR}/tuning_scores_intro.json")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


10/80 passages | 88 sentences | 3 s
20/80 passages | 169 sentences | 5 s
30/80 passages | 250 sentences | 6 s
40/80 passages | 334 sentences | 7 s
50/80 passages | 423 sentences | 9 s
60/80 passages | 499 sentences | 10 s
70/80 passages | 579 sentences | 11 s
80/80 passages | 655 sentences | 12 s
Saved 655 sentences to /content/drive/MyDrive/hallucination_week3/tuning_scores_intro.json | total time: 12 s


In [ ]:
import json, itertools
import numpy as np
from sklearn.metrics import roc_auc_score

rows = json.load(open(f"{SAVE_DIR}/tuning_scores_intro.json"))
y = np.array([r["label"] != "accurate" for r in rows])   # True = hallucinated
print("Sentences:", len(y), "| hallucinated:", int(y.sum()), f"({y.mean():.0%})")

def metrics(pred, y):
    tp = int((pred & y).sum()); fp = int((pred & ~y).sum()); fn = int((~pred & y).sum())
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return round(p, 2), round(r, 2), round(f, 2), int(pred.sum())

def verdict(row, ent_thr, con_thr, con_top, min_sim):
    ev = row["evidence"]
    max_ent = max((e["ent"] for e in ev), default=0)
    con = max((e["con"] for e in ev if e["rank"] < con_top and e["sim"] >= min_sim), default=0)
    if max_ent >= ent_thr: return "supported"
    if con >= con_thr: return "contradicted"
    return "neutral"

# baseline: say "hallucinated" for every sentence
base = metrics(np.ones(len(y), dtype=bool), y)
print("BASELINE (everything hallucinated): precision %.2f recall %.2f F1 %.2f\n" % base[:3])

# Predictor A: hallucinated = "contradicted"
results = []
for ent_thr, con_thr, con_top, min_sim in itertools.product(
        [0.5, 0.8, 0.9], [0.5, 0.8, 0.9, 0.95], [1, 2, 3, 5], [0.3, 0.5]):
    pred = np.array([verdict(r, ent_thr, con_thr, con_top, min_sim) == "contradicted" for r in rows])
    results.append(((ent_thr, con_thr, con_top, min_sim), metrics(pred, y)))

print("PREDICTOR A: hallucinated = Contradicted  (top 5 by F1)")
print("settings = (ent_thr, con_thr, con_top, min_sim)")
for s, m in sorted(results, key=lambda x: -x[1][2])[:5]:
    print(f"  {s} -> precision {m[0]}, recall {m[1]}, F1 {m[2]}, flagged {m[3]}")

print("\nPREDICTOR A: best precision where recall >= 0.20")
ok = [x for x in results if x[1][1] >= 0.20]
for s, m in sorted(ok, key=lambda x: -x[1][0])[:3]:
    print(f"  {s} -> precision {m[0]}, recall {m[1]}, F1 {m[2]}, flagged {m[3]}")

# Predictor B: hallucinated = anything not Supported
print("\nPREDICTOR B: hallucinated = not Supported")
for ent_thr in [0.5, 0.8, 0.9]:
    pred = np.array([verdict(r, ent_thr, 2, 1, 0.3) != "supported" for r in rows])
    m = metrics(pred, y)
    print(f"  ent_thr {ent_thr} -> precision {m[0]}, recall {m[1]}, F1 {m[2]}")

# Threshold-free check: how well do raw scores rank hallucinated above accurate?
s1 = [1 - max((e["ent"] for e in r["evidence"]), default=0) for r in rows]
s2 = [max((e["con"] for e in r["evidence"] if e["sim"] >= 0.3), default=0) for r in rows]
print("\nAUROC (0.5 = random guessing, 1.0 = perfect)")
print("  score = 1 - max entailment:", round(roc_auc_score(y, s1), 3))
print("  score = max contradiction :", round(roc_auc_score(y, s2), 3))

Sentences: 655 | hallucinated: 477 (73%)
BASELINE (everything hallucinated): precision 0.73 recall 1.00 F1 0.84

PREDICTOR A: hallucinated = Contradicted  (top 5 by F1)
settings = (ent_thr, con_thr, con_top, min_sim)
  (0.5, 0.5, 5, 0.3) -> precision 0.84, recall 0.42, F1 0.56, flagged 239
  (0.8, 0.5, 5, 0.3) -> precision 0.84, recall 0.42, F1 0.56, flagged 239
  (0.9, 0.5, 5, 0.3) -> precision 0.83, recall 0.42, F1 0.56, flagged 241
  (0.5, 0.5, 3, 0.3) -> precision 0.85, recall 0.39, F1 0.54, flagged 221
  (0.8, 0.5, 3, 0.3) -> precision 0.85, recall 0.39, F1 0.54, flagged 221

PREDICTOR A: best precision where recall >= 0.20
  (0.5, 0.9, 1, 0.5) -> precision 0.94, recall 0.22, F1 0.35, flagged 110
  (0.8, 0.9, 1, 0.5) -> precision 0.94, recall 0.22, F1 0.35, flagged 110
  (0.9, 0.9, 1, 0.5) -> precision 0.94, recall 0.22, F1 0.35, flagged 110

PREDICTOR B: hallucinated = not Supported
  ent_thr 0.5 -> precision 0.76, recall 0.99, F1 0.86
  ent_thr 0.8 -> precision 0.76, recall 1.0,

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
passages = np.array([r["passage"] for r in rows])
by_passage = {u: np.where(passages == u)[0] for u in np.unique(passages)}
keys = list(by_passage)

def fpr(pred):
    return round(float((pred & ~y).sum() / (~y).sum()), 3)

settings = {
    "A high-precision": (0.8, 0.9, 1, 0.5),
    "A balanced":       (0.8, 0.5, 5, 0.3),
}
for name, s in settings.items():
    pred = np.array([verdict(r, *s) == "contradicted" for r in rows])
    p, rec, f, n = metrics(pred, y)
    precs = []
    for _ in range(1000):
        pick = rng.choice(len(keys), size=len(keys), replace=True)
        idx = np.concatenate([by_passage[keys[i]] for i in pick])
        pp, yy = pred[idx], y[idx]
        if pp.sum():
            precs.append((pp & yy).sum() / pp.sum())
    lo, hi = np.percentile(precs, [2.5, 97.5])
    print(f"{name}: precision {p} (95% CI {lo:.2f}-{hi:.2f}), recall {rec}, "
          f"false-alarm rate {fpr(pred)}, flagged {n}")

print("\nPredictor B with low entailment cutoffs")
for t in [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]:
    pred = np.array([verdict(r, t, 2, 1, 0.3) != "supported" for r in rows])
    p, rec, f, n = metrics(pred, y)
    print(f"  ent_thr {t}: precision {p}, recall {rec}, F1 {f}, false-alarm rate {fpr(pred)}")

A high-precision: precision 0.94 (95% CI 0.88-0.98), recall 0.22, false-alarm rate 0.039, flagged 110
A balanced: precision 0.84 (95% CI 0.77-0.90), recall 0.42, false-alarm rate 0.213, flagged 239

Predictor B with low entailment cutoffs
  ent_thr 0.02: precision 0.78, recall 0.93, F1 0.85, false-alarm rate 0.708
  ent_thr 0.05: precision 0.77, recall 0.97, F1 0.86, false-alarm rate 0.764
  ent_thr 0.1: precision 0.77, recall 0.99, F1 0.87, false-alarm rate 0.781
  ent_thr 0.2: precision 0.77, recall 0.99, F1 0.87, false-alarm rate 0.798
  ent_thr 0.3: precision 0.77, recall 0.99, F1 0.87, false-alarm rate 0.803
  ent_thr 0.5: precision 0.76, recall 0.99, F1 0.86, false-alarm rate 0.826


In [ ]:
import json, numpy as np
from sklearn.metrics import roc_auc_score

# 1. Freeze settings BEFORE looking at test data
FROZEN = {"A high-precision": [0.8, 0.9, 1, 0.5],
          "A balanced":       [0.8, 0.5, 5, 0.3]}
json.dump(FROZEN, open(f"{SAVE_DIR}/frozen_settings.json", "w"))

# 2. Score the test passages (once)
split = json.load(open(f"{SAVE_DIR}/split.json"))
test_rows = score_passages(split["test"], f"{SAVE_DIR}/test_scores_intro.json")

# 3. Evaluate with the frozen settings
def verdict(row, ent_thr, con_thr, con_top, min_sim):
    ev = row["evidence"]
    max_ent = max((e["ent"] for e in ev), default=0)
    con = max((e["con"] for e in ev if e["rank"] < con_top and e["sim"] >= min_sim), default=0)
    if max_ent >= ent_thr: return "supported"
    if con >= con_thr: return "contradicted"
    return "neutral"

y = np.array([r["label"] != "accurate" for r in test_rows])
passages = np.array([r["passage"] for r in test_rows])
by_p = {u: np.where(passages == u)[0] for u in np.unique(passages)}
keys = list(by_p)
rng = np.random.default_rng(0)

def prf(pred, yy):
    tp = (pred & yy).sum(); fp = (pred & ~yy).sum(); fn = (~pred & yy).sum()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    far = fp / (~yy).sum()
    return p, r, far

print(f"Test sentences: {len(y)} | hallucinated: {int(y.sum())} ({y.mean():.0%})")
print(f"BASELINE (everything hallucinated): precision {y.mean():.2f}, recall 1.00, false-alarm rate 1.00\n")

for name, s in FROZEN.items():
    pred = np.array([verdict(r, *s) == "contradicted" for r in test_rows])
    p, r, far = prf(pred, y)
    ps = []
    for _ in range(1000):
        idx = np.concatenate([by_p[keys[i]] for i in rng.choice(len(keys), len(keys), replace=True)])
        if pred[idx].sum():
            ps.append((pred[idx] & y[idx]).sum() / pred[idx].sum())
    lo, hi = np.percentile(ps, [2.5, 97.5])
    print(f"{name}: precision {p:.2f} (95% CI {lo:.2f}-{hi:.2f}), recall {r:.2f}, "
          f"false-alarm rate {far:.3f}, flagged {int(pred.sum())}")

s1 = [1 - max((e["ent"] for e in r["evidence"]), default=0) for r in test_rows]
s2 = [max((e["con"] for e in r["evidence"] if e["sim"] >= 0.3), default=0) for r in test_rows]
print("\nAUROC  1-max entailment:", round(roc_auc_score(y, s1), 3),
      "| max contradiction:", round(roc_auc_score(y, s2), 3))

10/158 passages | 80 sentences | 3 s
20/158 passages | 161 sentences | 5 s
30/158 passages | 250 sentences | 6 s
40/158 passages | 331 sentences | 7 s
50/158 passages | 416 sentences | 8 s
60/158 passages | 485 sentences | 10 s
70/158 passages | 561 sentences | 11 s
80/158 passages | 636 sentences | 12 s
90/158 passages | 708 sentences | 14 s
100/158 passages | 784 sentences | 15 s
110/158 passages | 866 sentences | 16 s
120/158 passages | 939 sentences | 17 s
130/158 passages | 1013 sentences | 18 s
140/158 passages | 1097 sentences | 19 s
150/158 passages | 1185 sentences | 21 s
Saved 1253 sentences to /content/drive/MyDrive/hallucination_week3/test_scores_intro.json | total time: 22 s
Test sentences: 1253 | hallucinated: 915 (73%)
BASELINE (everything hallucinated): precision 0.73, recall 1.00, false-alarm rate 1.00

A high-precision: precision 0.90 (95% CI 0.86-0.94), recall 0.21, false-alarm rate 0.059, flagged 208
A balanced: precision 0.86 (95% CI 0.82-0.90), recall 0.44, false-